In [33]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/br/qhn11dt91tvcg9cqgczhjdlr0000gn/T/pip-build-env-_njv569o/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider remo

In [34]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

In [35]:
SEED = 128
df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")

In [36]:
df_sample = df.sample(50000, random_state=SEED)  # 50k rijen

train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED, 
)

reg = setup(
    data=train_df,
    target="england_wales_demand",
    session_id=SEED,
    fold=5,
    verbose=True,
)

best_model = compare_models(sort="MAE", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="MAE", 
    fold=5,
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

save_model(final_model, "../Model_ElecDemand/england_wales_demand_predictor")

,Description,Value
0,Session id,128
1,Target,england_wales_demand
2,Target type,Regression
3,Original data shape,"(35000, 15)"
4,Transformed data shape,"(35000, 15)"
5,Transformed train set shape,"(24500, 15)"
6,Transformed test set shape,"(10500, 15)"
7,Numeric features,13
8,Categorical features,1
9,Preprocess,True


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,2167.3458,9061167.7391,3009.6872,0.8258,0.0937,0.0684,0.2660
rf,Random Forest Regressor,2213.0787,9253854.2962,3041.6253,0.8221,0.0953,0.0702,0.5880
lightgbm,Light Gradient Boosting Machine,2264.2503,8944293.3384,2990.3236,0.8280,0.0932,0.0715,0.5440
gbr,Gradient Boosting Regressor,2569.5163,10934840.8432,3306.5145,0.7897,0.1037,0.0816,0.3780
dt,Decision Tree Regressor,2628.6410,13430900.9416,3664.5523,0.7417,0.1153,0.0835,0.2260
knn,K Neighbors Regressor,2809.8565,14150537.3267,3761.4987,0.7279,0.1162,0.0886,0.0220
ada,AdaBoost Regressor,2965.3517,13512052.1595,3675.5675,0.7402,0.1193,0.0976,0.1780
lar,Least Angle Regression,3298.7231,17556078.4241,4189.8416,0.6624,0.1317,0.1056,0.0100
ridge,Ridge Regression,3308.5516,17627231.3731,4198.3304,0.6611,0.1319,0.1060,0.2100
lr,Linear Regression,3308.5516,17627230.9297,4198.3304,0.6611,0.1319,0.1060,0.3600


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,2709.7719,12461782.4707,3530.1250,0.7633,0.1091,0.0850
1,2632.0786,11813698.2783,3437.1061,0.7712,0.1062,0.0826
2,2753.8141,12752904.8076,3571.1209,0.7522,0.1116,0.0874
3,2686.5811,12290100.3764,3505.7239,0.7642,0.1092,0.0851
4,2735.4814,12617972.4330,3552.1785,0.7580,0.1104,0.0864
Mean,2703.5454,12387291.6732,3519.2509,0.7618,0.1093,0.0853
Std,42.3814,325849.9614,46.5268,0.0064,0.0018,0.0016


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,1624.9342,5134703.5696,2265.9884,0.9029,0.0702,0.0510


Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['settlement_period',
                                              'embedded_wind_generation',
                                              'embedded_wind_capacity',
                                              'embedded_solar_generation',
                                              'embedded_solar_capacity',
                                              'non_bm_stor',
                                              'pump_storage_pumping',
                                              'ifa2_flow', 'britned_flow',
                                              'moyle_flow', 'east_west_flow',
                                              'nemo_flow', 'year'],
                                     transformer=Simp...puter())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['settlement_date'],
                                  

In [37]:
from pycaret.classification import load_model
import pandas as pd

# Load model
model = load_model("../Model_ElecDemand/england_wales_demand_predictor")

# Extract features the model expects
expected_features = model[0].feature_names_in_

print("Model expects", len(expected_features), "features:")
for col in expected_features:
    print(col)

df_sample.columns


Transformation Pipeline and Model Successfully Loaded
Model expects 15 features:
settlement_date
settlement_period
embedded_wind_generation
embedded_wind_capacity
embedded_solar_generation
embedded_solar_capacity
non_bm_stor
pump_storage_pumping
ifa2_flow
britned_flow
moyle_flow
east_west_flow
nemo_flow
year
england_wales_demand


Index(['settlement_date', 'settlement_period', 'england_wales_demand',
       'embedded_wind_generation', 'embedded_wind_capacity',
       'embedded_solar_generation', 'embedded_solar_capacity', 'non_bm_stor',
       'pump_storage_pumping', 'ifa2_flow', 'britned_flow', 'moyle_flow',
       'east_west_flow', 'nemo_flow', 'year'],
      dtype='object')

In [38]:
print(val_df.head(5))


       settlement_date  settlement_period  england_wales_demand  \
27967       2002-08-06                 34                 36859   
93823       2006-05-09                 34                 39759   
1714        2001-02-05                 35                 47592   
228306      2014-01-09                 19                 39437   
268672      2016-04-29                 19                 31851   

        embedded_wind_generation  embedded_wind_capacity  \
27967                 783.757098             2544.300517   
93823                 783.757098             2544.300517   
1714                  783.757098             2544.300517   
228306                998.000000             2434.000000   
268672               2487.000000             4260.000000   

        embedded_solar_generation  embedded_solar_capacity  non_bm_stor  \
27967                  480.254956              4536.541419            0   
93823                  480.254956              4536.541419            0   
1714       